# Berry Picker Assistant – Zone grid (Jyväskylä demo)
**Team Picky · ML team · TIES4570 CSDP**

**Goal:** turn any coordinate (lat, lon) + date + berry into a 0–100 picking score.

**Data:** Luke multi-source forest inventory maps, MS-NFI 2023 (16 m pixels, CC BY 4.0), read from the funet mirror:
- `paatyyppi` – site main class (mineral soil / spruce mire / pine mire / open bog)
- `kasvupaikka` – site fertility class (herb-rich … xeric)
- `ika` – stand age (0–10 years ≈ clear-cut / young stand)

**Steps:** read a 40 × 40 km window around Jyväskylä → classify every 16 m pixel into a forest type → summarise into 256 m cells → berry potential per cell → `zone_scorer.py` with `score_at(lat, lon, date, berry)`.

In [ ]:
!pip install -q rasterio pyproj
import os, math, json, shutil, numpy as np, pandas as pd, rasterio, requests
from rasterio.windows import from_bounds
from pyproj import Transformer
BASE = "https://mirrors.nic.funet.fi/index/geodata/luke/vmi/2023/"
CENTER_LAT, CENTER_LON = 62.2426, 25.7473      # Jyväskylä city centre
HALF_M = 20000                                  # 20 km each way -> 40 x 40 km
BLOCK = 16                                      # 16 pixels x 16 m = 256 m cells
to_tm = Transformer.from_crs(4326, 3067, always_xy=True)
to_ll = Transformer.from_crs(3067, 4326, always_xy=True)
cx, cy = to_tm.transform(CENTER_LON, CENTER_LAT)
print("Jyväskylä in ETRS-TM35FIN:", round(cx), round(cy))

## 1. Read only the Jyväskylä window from each national map

In [ ]:
def read_window(name):
    url = BASE + f"{name}_vmi1x_1923.tif"
    def _read(path):
        with rasterio.open(path) as src:
            win = from_bounds(cx - HALF_M, cy - HALF_M, cx + HALF_M, cy + HALF_M, src.transform)
            win = win.round_offsets().round_lengths()
            arr = src.read(1, window=win)
            return arr, src.window_transform(win), src.crs
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif",
                          GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="2"):
            out = _read("/vsicurl/" + url)
        print(f"{name}: read window over HTTP", out[0].shape)
    except Exception as e:
        print(f"{name}: window read failed ({e}); downloading the whole file…")
        local = f"/tmp/{name}.tif"
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(local, "wb") as f:
                for chunk in r.iter_content(1 << 22): f.write(chunk)
        out = _read(local); os.remove(local)
        print(f"{name}: read from download", out[0].shape)
    return out

pt, tr, crs = read_window("paatyyppi")
kp, tr2, _ = read_window("kasvupaikka")
ika, tr3, _ = read_window("ika")
assert tr == tr2 == tr3, "rasters are not on the same grid"
print(crs, tr)
for n, a in [("paatyyppi", pt), ("kasvupaikka", kp)]:
    v, c = np.unique(a, return_counts=True); print(n, dict(zip(v.tolist(), c.tolist())))

## 2. Classify each 16 m pixel into a forest type
| Code | Forest type | Rule |
|---|---|---|
| 0 | non_forest | no forest data (fields, water, built-up) |
| 1 | herb_rich | mineral soil, fertility 1–2 |
| 2 | mesic_heath | mineral soil, fertility 3 (best bilberry) |
| 3 | sub_xeric | mineral soil, fertility 4 (best lingonberry) |
| 4 | xeric | mineral soil, fertility 5–10 |
| 5 | peatland | spruce mire, pine mire or open bog (cloudberry) |
| 6 | clearcut | mineral soil, stand age 0–10 years |

In [ ]:
TYPES = ["non_forest", "herb_rich", "mesic_heath", "sub_xeric", "xeric", "peatland", "clearcut"]
ft = np.zeros(pt.shape, np.uint8)
mineral = pt == 1
ft[mineral & np.isin(kp, [1, 2])] = 1
ft[mineral & (kp == 3)] = 2
ft[mineral & (kp == 4)] = 3
ft[mineral & (kp >= 5) & (kp <= 10)] = 4
ft[(pt >= 2) & (pt <= 4)] = 5
ft[mineral & (ika >= 0) & (ika <= 10)] = 6
share = pd.Series(np.bincount(ft.ravel(), minlength=7) / ft.size, index=TYPES)
print((share * 100).round(1).to_string())

## 3. Summarise into 256 m cells and compute berry potential

In [ ]:
# Typical yield of an average year by forest type (kg/ha) – same table as predict_yield.py
SITE_POTENTIAL = {
    "bilberry":    {"herb_rich": 5, "mesic_heath": 25, "sub_xeric": 15, "xeric": 5,  "peatland": 10, "clearcut": 2},
    "lingonberry": {"herb_rich": 2, "mesic_heath": 10, "sub_xeric": 30, "xeric": 20, "peatland": 10, "clearcut": 25},
    "cloudberry":  {"herb_rich": 0, "mesic_heath": 0,  "sub_xeric": 0,  "xeric": 0,  "peatland": 15, "clearcut": 0},
}
H, W = (ft.shape[0] // BLOCK) * BLOCK, (ft.shape[1] // BLOCK) * BLOCK
blocks = ft[:H, :W].reshape(H // BLOCK, BLOCK, W // BLOCK, BLOCK)
rows, cols = H // BLOCK, W // BLOCK
frac = {t: (blocks == k).mean(axis=(1, 3)) for k, t in enumerate(TYPES)}

ii, jj = np.meshgrid(np.arange(rows), np.arange(cols), indexing="ij")
e = tr.c + (jj * BLOCK + BLOCK / 2) * tr.a
n = tr.f + (ii * BLOCK + BLOCK / 2) * tr.e
lon, lat = to_ll.transform(e.ravel(), n.ravel())
G = pd.DataFrame({"cell_id": [f"JKL_{r:03d}_{c:03d}" for r, c in zip(ii.ravel(), jj.ravel())],
                  "lat": np.round(lat, 6), "lon": np.round(lon, 6),
                  "e_tm35": e.ravel().round(1), "n_tm35": n.ravel().round(1)})
for t in TYPES: G[f"frac_{t}"] = frac[t].ravel().round(3)
G["main_type"] = np.array(TYPES)[np.argmax(np.stack([frac[t] for t in TYPES]), axis=0).ravel()]
for berry, pot in SITE_POTENTIAL.items():    # 0-1: share-weighted potential vs the best forest type
    best = max(pot.values())
    G[f"site_{berry}"] = sum(G[f"frac_{t}"] * v for t, v in pot.items()).div(best).round(3)
print(G.shape); print(G.main_type.value_counts().to_string())
G[[c for c in G.columns if c.startswith("site_")]].describe().round(3)

## 4. Preview maps (darker = better ground for that berry)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
for a, berry, cmap in zip(ax, SITE_POTENTIAL, ["Blues", "Reds", "Oranges"]):
    im = a.imshow(G[f"site_{berry}"].values.reshape(rows, cols), cmap=cmap, vmin=0, vmax=1)
    a.plot(cols / 2, rows / 2, "k*", ms=14); a.set_title(f"{berry} – site potential (★ = Jyväskylä)")
    a.axis("off"); plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.savefig("/kaggle/working/zone_preview.png", dpi=110); plt.show()

## 5. `zone_scorer.py` – the function the app calls

In [ ]:
%%writefile /kaggle/working/zone_scorer.py
"""
zone_scorer.py - lat/lon -> berry picking score (Berry Picker Assistant, Team Picky ML team)

    from zone_scorer import ZoneScorer
    zs = ZoneScorer("zone_grid.csv")
    zs.score_at(62.20, 25.80, "2026-07-25", "bilberry")
    zs.top_zones("bilberry", "2026-07-25", n=10)

score = 100 x site potential (forest mix of the 256 m cell) x ripeness x year quality (0.8 in a normal year)
Needs only numpy + pandas.
"""
import datetime as _dt
import math
import numpy as np
import pandas as pd

# Ripening windows (day of year) at latitude 62 N (Jyvaskyla); ~2 days later per degree further north
RIPENING = {"bilberry": (190, 240), "lingonberry": (225, 280), "cloudberry": (185, 215)}
DAYS_PER_DEGREE_NORTH = 2.0
CELL_HALF_DIAGONAL_M = 185   # a 256 m cell; farther away than this = outside the grid


def _doy(date):
    if isinstance(date, str):
        date = _dt.date.fromisoformat(date[:10])
    return date.timetuple().tm_yday


def ripeness(berry, day_of_year, lat=62.0):
    shift = DAYS_PER_DEGREE_NORTH * (lat - 62.0)
    a, b = RIPENING[berry][0] + shift, RIPENING[berry][1] + shift
    if a <= day_of_year <= b:
        return 1.0, "ripe"
    if a - 10 <= day_of_year < a:
        return 0.5, "ripening soon"
    if b < day_of_year <= b + 10:
        return 0.5, "end of season"
    return 0.0, "out of season"


class ZoneScorer:
    def __init__(self, grid_csv="zone_grid.csv"):
        self.g = pd.read_csv(grid_csv)
        self._lat = np.radians(self.g["lat"].to_numpy())
        self._lon = np.radians(self.g["lon"].to_numpy())

    def _nearest(self, lat, lon):
        la, lo = math.radians(lat), math.radians(lon)
        d2 = (self._lat - la) ** 2 + ((self._lon - lo) * math.cos(la)) ** 2
        i = int(d2.argmin())
        return i, math.sqrt(d2[i]) * 6371000

    def score_at(self, lat, lon, date, berry, year_index=1.0):
        """Score 0-100 for one point. year_index 1.0 = normal year (from predict_yield.py)."""
        i, dist = self._nearest(lat, lon)
        if dist > CELL_HALF_DIAGONAL_M:
            return {"score": None, "reason": "outside the demo grid (Jyvaskyla 40x40 km)"}
        c = self.g.iloc[i]
        ripe, status = ripeness(berry, _doy(date), lat)
        site = float(c[f"site_{berry}"])
        score = 100 * site * ripe * min(year_index, 2.0) / 2.0 * 1.6
        mix = {k[5:]: float(v) for k, v in c.items() if k.startswith("frac_") and v >= 0.05}
        return {"score": round(max(0.0, min(100.0, score)), 1), "berry": berry, "cell_id": c["cell_id"],
                "cell_center": [float(c["lat"]), float(c["lon"])], "main_forest_type": c["main_type"],
                "forest_mix": mix, "site_potential": site, "season": status, "year_index": year_index}

    def top_zones(self, berry, date, n=10, year_index=1.0):
        """Best n cells for a berry on a date (for the map's 'go here' list)."""
        doy = _doy(date)
        ripe = np.array([ripeness(berry, doy, la)[0] for la in self.g["lat"]])
        s = 100 * self.g[f"site_{berry}"].to_numpy() * ripe * min(year_index, 2.0) / 2.0 * 1.6
        top = np.argsort(-s)[:n]
        return [{"cell_id": self.g.at[i, "cell_id"], "lat": float(self.g.at[i, "lat"]),
                 "lon": float(self.g.at[i, "lon"]), "score": round(float(s[i]), 1),
                 "main_forest_type": self.g.at[i, "main_type"]} for i in top]


## 6. Package + test with real coordinates

In [ ]:
H_DIR = "/kaggle/working/zone_handoff"; os.makedirs(H_DIR, exist_ok=True)
G.to_csv(f"{H_DIR}/zone_grid.csv", index=False)
shutil.copy("/kaggle/working/zone_scorer.py", H_DIR); shutil.copy("/kaggle/working/zone_preview.png", H_DIR)
import sys; sys.path.insert(0, H_DIR)
from zone_scorer import ZoneScorer
zs = ZoneScorer(f"{H_DIR}/zone_grid.csv")
tests = [("Jyväskylä centre", 62.2426, 25.7473), ("Laajavuori forest", 62.2595, 25.6880),
         ("Muurame forest", 62.1300, 25.6700), ("Far outside (Helsinki)", 60.17, 24.94)]
for name, la, lo in tests:
    for berry, date in [("bilberry", "2026-07-25"), ("lingonberry", "2026-08-30")]:
        print(f"{name:24s} {berry:11s} {date}:", json.dumps(zs.score_at(la, lo, date, berry), ensure_ascii=False))
print("\nTop 5 bilberry zones on 25 July:")
for z in zs.top_zones("bilberry", "2026-07-25", n=5): print(z)

In [ ]:
readme = f'''# Zone grid – lat/lon → berry score (Jyväskylä demo) · Team Picky ML

## Files
| File | What |
|---|---|
| `zone_grid.csv` | {len(G):,} cells of 256 m × 256 m covering 40 × 40 km around Jyväskylä |
| `zone_scorer.py` | `ZoneScorer.score_at(lat, lon, date, berry)` and `top_zones(berry, date, n)` |
| `zone_preview.png` | maps of the site potential per berry |

## Install / use
```
pip install numpy pandas
```
```python
from zone_scorer import ZoneScorer
zs = ZoneScorer("zone_grid.csv")
zs.score_at(62.13, 25.67, "2026-07-25", "bilberry")
# -> {{"score": .., "main_forest_type": .., "forest_mix": {{..}}, "season": "ripe", ..}}
```

## How the score is made
score (0–100) = 100 × site potential × ripeness × year quality (normal year = 0.8)
- **site potential (0–1)** = forest-type mix of the cell × typical kg/ha of each type ÷ best type
  (bilberry: mesic heath; lingonberry: sub-xeric pine forest; cloudberry: peatland)
- **ripeness** = 1 inside the ripening window, 0.5 within 10 days of it, else 0; windows shift ~2 days later per degree north
- **year quality** = 1.0 (normal year); weather models did not beat this in testing (see yield_handoff)

## grid columns
cell_id, lat, lon (cell centre, WGS84), e_tm35, n_tm35 (EPSG:3067), frac_<type> (share of 16 m pixels),
main_type, site_bilberry, site_lingonberry, site_cloudberry

## Data source
Luke MS-NFI 2023 (paatyyppi, kasvupaikka, ika), 16 m, CC BY 4.0 – © Natural Resources Institute Finland (Luke).
To cover another area: change CENTER_LAT/CENTER_LON in the BPA Zone Grid notebook and re-run.
'''
open(f"{H_DIR}/README.md", "w").write(readme)
shutil.rmtree(f"{H_DIR}/__pycache__", ignore_errors=True)
shutil.make_archive("/kaggle/working/zone_handoff", "zip", H_DIR)
!ls -la /kaggle/working/zone_handoff; du -sh /kaggle/working/zone_handoff.zip